in this part we will try 3 models acrossboth datasets
to understand which algorithm works best and why



In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, DataStructs
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor, RandomForestClassifier,
    GradientBoostingRegressor, GradientBoostingClassifier
)
from sklearn.metrics import roc_auc_score

print("imports done")

imports done


In [48]:
def smiles_to_fingerprints(smiles_list):
    fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    features = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            fp = fp_gen.GetFingerprint(mol)
            arr = np.zeros((2048,))
            DataStructs.ConvertToNumpyArray(fp, arr)
            features.append(arr)
        else:
            features.append(np.zeros(2048))
    return np.array(features)

print("function defined")

function defined


In [49]:
lipo_url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/Lipophilicity.csv"
df_lipo = pd.read_csv(lipo_url)

X_lipo = smiles_to_fingerprints(df_lipo['smiles'])
y_lipo = df_lipo['exp'].values

print("Lipophilicity X shape:", X_lipo.shape)
print("Lipophilicity y shape:", y_lipo.shape)

Lipophilicity X shape: (4200, 2048)
Lipophilicity y shape: (4200,)


In [50]:
url = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/BBBP.csv"
df_bbbp= pd.read_csv(url)
X_bbbp = smiles_to_fingerprints(df_bbbp['smiles'])
y_bbbp = df_bbbp['p_np'].values
print("Shape x:", X_bbbp.shape)
print("Shape y:", y_bbbp.shape)

[15:50:18] Explicit valence for atom # 1 N, 4, is greater than permitted
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] Explicit valence for atom # 6 N, 4, is greater than permitted
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] Explicit valence for atom # 6 N, 4, is greater than permitted
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] Explicit valence for atom # 11 N, 4, is greater than pe

Shape x: (2050, 2048)
Shape y: (2050,)


[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not removing hydrogen atom without neighbors
[15:50:18] WARNING: not r

In [51]:
X_train_lipo, X_test_lipo, y_train_lipo, y_test_lipo = train_test_split(
    X_lipo, y_lipo, test_size=0.2, random_state=42
)

print("Training samples:", X_train_lipo.shape[0])
print("Testing samples:", X_test_lipo.shape[0])

Training samples: 3360
Testing samples: 840


In [52]:
X_train_bbbp, X_test_bbbp, y_train_bbbp, y_test_bbbp = train_test_split(
    X_bbbp, y_bbbp, test_size=0.2, random_state=42, stratify=y_bbbp
)

print("Training samples:", X_train_bbbp.shape[0])
print("Testing samples:", X_test_bbbp.shape[0])

# Verify balance is maintained
unique, counts = np.unique(y_test_bbbp, return_counts=True)
print("Test set class distribution:", dict(zip(unique, counts)))

Training samples: 1640
Testing samples: 410
Test set class distribution: {np.int64(0): np.int64(97), np.int64(1): np.int64(313)}


In [53]:
lipo_results = {}

In [54]:
lr = LinearRegression()
lr.fit(X_train_lipo, y_train_lipo)
lr_preds = lr.predict(X_test_lipo)
lr_rmse = np.sqrt(np.mean((y_test_lipo - lr_preds) ** 2))

lipo_results["Linear Regression"] = lr_rmse
print("Linear Regression RMSE:", round(lr_rmse, 4))

Linear Regression RMSE: 1.2758


In [55]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_lipo, y_train_lipo)

rf_preds = rf.predict(X_test_lipo)

rf_rmse = np.sqrt(np.mean((y_test_lipo - rf_preds) ** 2))
print("Random Forest RMSE for lipo:", round(rf_rmse, 4))

Random Forest RMSE for lipo: 0.8252


In [56]:
gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train_lipo, y_train_lipo)
gb_preds = gb.predict(X_test_lipo)
gb_rmse = np.sqrt(np.mean((y_test_lipo - gb_preds) ** 2))

lipo_results["Gradient Boosting"] = gb_rmse
print("Gradient Boosting RMSE:", round(gb_rmse, 4))

Gradient Boosting RMSE: 0.927


random forest outperformed gradient boosting here although the fact that gradient boosting is better 

possible reason could be its prone to overfitting on high-dimensional sparse data like 2048-bit fingerprints, especially with default hyperparameters. It builds trees that aggressively chase training error reduction, which can hurt generalization to unseen molecules. Random Forest's random subsampling (of both data and features) makes it more robust on this type of data without tuning.


In [57]:
bbbp_results = {}

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_bbbp, y_train_bbbp)
logreg_probs = logreg.predict_proba(X_test_bbbp)[:, 1]
logreg_auc = roc_auc_score(y_test_bbbp, logreg_probs)

bbbp_results["Logistic Regression"] = logreg_auc
print("Logistic Regression AUC:", round(logreg_auc, 4))

Logistic Regression AUC: 0.9083


In [58]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_bbbp, y_train_bbbp)
rf_clf_probs = rf_clf.predict_proba(X_test_bbbp)[:, 1]
rf_clf_auc = roc_auc_score(y_test_bbbp, rf_clf_probs)

bbbp_results["Random Forest"] = rf_clf_auc
print("Random Forest AUC:", round(rf_clf_auc, 4))

Random Forest AUC: 0.9156


In [59]:
gb_clf = GradientBoostingClassifier(random_state=42)
gb_clf.fit(X_train_bbbp, y_train_bbbp)
gb_clf_probs = gb_clf.predict_proba(X_test_bbbp)[:, 1]
gb_clf_auc = roc_auc_score(y_test_bbbp, gb_clf_probs)

bbbp_results["Gradient Boosting"] = gb_clf_auc
print("Gradient Boosting AUC:", round(gb_clf_auc, 4))

Gradient Boosting AUC: 0.9003


logistic regression performed as well as random forest on this task.
This suggests BBBP classification may be a somewhat easier separation problem than predicting exact LogP values, where a simpler decision boundary on the fingerprint bits is sufficient.
